In [ ]:
import os
import sys

import polars as pl

sys.path.append("/workspace")

from drsd.reader import MINDsmallReader

pl.Config(tbl_rows=5)
os.makedirs("/workspace/processed", exist_ok=True)

In [ ]:
reader = MINDsmallReader()

### Target News

In [ ]:
behavior_df = pl.concat([reader.get_behavior_df("train"), reader.get_behavior_df("dev")]).with_columns(
    pl.col("time").str.strptime(pl.Datetime, "%m/%d/%Y %r")
)
behavior_df

### Extract Time

In [ ]:
output_df = (
    behavior_df.select(pl.col("clicked").alias("news_id"), "time")
    .explode("news_id")
    .group_by("news_id")
    .agg(pl.col("time").min())
    .sort("news_id")
)
output_df

### Output DataFrame

In [ ]:
output_df.write_parquet("/workspace/processed/news_time.parquet")